In [54]:
import pandas as pd 
import sklearn as sk
import os
import pandas as pd
import numpy as np 
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import scanpy as sc
import anndata as ad
import bbknn
from sklearn.decomposition import PCA
import numpy as np
import harmonypy as hm
GENE_PANEL = ["ATOH1","DLL1","DLL4","GFI1","AREG","HES1","HES5","JAG2","NOTCH1","NOTCH2","NOTCH3",
              "OLFM4","LEF1","APCDD1","WNT6","NEUROG3","NEUROD1","KRT20","NEURL1","LGR5"]

In [ ]:
data_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/saver_mat'
train_id=['CRC0322','CRC0327','CRC0542','CRC0069']
tratt_cercato=['cetux','CTX72h','NT','NT72h']
train=pd.DataFrame()
for file in os.listdir(data_path):
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]
    if sample_name in train_id and trattamento in tratt_cercato:
        print(file)
        data=pd.read_csv(os.path.join(data_path,file),header=0,index_col=0)
        data=data.T
        data['sample']=sample_name
        data['cell_id']=data.index
        data['trattamento']=trattamento
        data.reset_index(drop=True,inplace=True)
        train=pd.concat([train,data])

filtered_annotated_saver_ribomito_CRC0322_cetux_1_log2_pc1_cpm.csv
filtered_annotated_saver_ribomito_CRC0322_NT_1_3000_log2_pc1_cpm.csv
filtered_annotated_saver_ribomito_CRC0327_cetux_2_log2_pc1_cpm.csv


In [52]:
from func_NT_cetux import *
n_hvg=1500
df=train
df_clean, dup = strip_prefix_from_genes(df, meta_cols=("cell_id","sample"), sep=":", on_duplicate="first")

# 2) AnnData + raw
adata_full = make_anndata_from_df(df_clean, set_raw=True)

# 3) cell cycle
score_cell_cycle(adata_full)

# 4) HVG + PCA
select_hvg_cell_ranger(adata_full, n_top_genes=n_hvg, batch_key="sample", subset=True)
scale_and_pca(adata_full, n_comps=42, max_value=10, random_state=42)
adata_base = adata_full

BEST = dict(
    n_pcs=20,          # PC da usare per BBKNN
    nwb=5,             # neighbors_within_batch
    trim=13,         # None oppure un intero (es. 12)
    leiden_res=0.3,    # granularità (resolution) per Leiden
    out_root="./bbknn_PPH"
)


# cartella output
def safe(s): return re.sub(r"[^A-Za-z0-9_=.,+-]", "_", str(s))
outdir = os.path.join(
    BEST["out_root"],
    f"npcs={BEST['n_pcs']}__nwb={BEST['nwb']}__trim={BEST['trim']}__res={BEST['leiden_res']}"
)
os.makedirs(outdir, exist_ok=True)
sc.settings.figdir = outdir

# 5)integrazione
adx = adata_base.copy()
bbknn.bbknn(
    adx,
    batch_key="sample",
    neighbors_within_batch=BEST['nwb'],
    trim=BEST["trim"],
    n_pcs=BEST["n_pcs"]
)


In [53]:
sc.tl.umap(adx, min_dist=0.2, random_state=42)  # UMAP usa il grafo BBKNN
save_umap_gene_panel(adx, outdir, GENE_PANEL)
# 6)clustering (Leiden)
clu_key = f"leiden_r{BEST['leiden_res']}"
sc.tl.leiden(adx, resolution=BEST["leiden_res"], key_added=clu_key, random_state=42)

# plot rapidi &
sc.pl.umap(adx, color=["sample", clu_key], ncols=2, wspace=0.3, frameon=False,
           show=False, save="_sample+cluster.png")
if "phase" in adx.obs:
    sc.pl.umap(adx, color=["phase"], frameon=False, show=False, save="_phase.png")

saving figure to file bbknn_PPH/npcs=20__nwb=5__trim=13__res=0.3/umap_genes_panel.png
saving figure to file bbknn_PPH/npcs=20__nwb=5__trim=13__res=0.3/umap_sample+cluster.png
saving figure to file bbknn_PPH/npcs=20__nwb=5__trim=13__res=0.3/umap_phase.png


In [6]:
data_obs = pd.DataFrame(adx.obs)

#aggiungo umap alle obs perche di default sta nel obsm
data_obs["umap_1"] = adx.obsm["X_umap"][:, 0]
data_obs["umap_2"] = adx.obsm["X_umap"][:, 1]

# Salva tutto in un CSV
data_obs.to_csv(
    path_or_buf="bbknn_PPH/npcs=20__nwb=3__trim=None__res=0.3/obs_data_with_umap.csv",
    index=False
)

In [19]:
data_obs['trattamento_sample'] = data_obs['sample'].astype(str) + '_' + data_obs['trattamento'].astype(str)

In [44]:
data_obs=data_obs.rename(columns={'umap_1':'x','umap_2':'y'})

In [45]:
data_obs

,cell_id,sample,trattamento,uid,S_score,G2M_score,phase,leiden_r0.3,x,y,trattamento_sample
uid,,,,,,,,,,,
CRC0322cetux|AAACCCACACACCAGC.1|cetux,AAACCCACACACCAGC.1,CRC0322,cetux,CRC0322cetux|AAACCCACACACCAGC.1|cetux,-1.195978,-2.071868,G1,0,4.587066,13.253962,CRC0322_cetux
CRC0322cetux|AAACCCACACTGGCCA.1|cetux,AAACCCACACTGGCCA.1,CRC0322,cetux,CRC0322cetux|AAACCCACACTGGCCA.1|cetux,-1.090331,-2.021245,G1,0,4.652971,12.768559,CRC0322_cetux
CRC0322cetux|AAACCCAGTGTCTTCC.1|cetux,AAACCCAGTGTCTTCC.1,CRC0322,cetux,CRC0322cetux|AAACCCAGTGTCTTCC.1|cetux,-1.364517,-2.205583,G1,3,4.026482,7.261326,CRC0322_cetux
CRC0322cetux|AAACGAATCGAAGCCC.1|cetux,AAACGAATCGAAGCCC.1,CRC0322,cetux,CRC0322cetux|AAACGAATCGAAGCCC.1|cetux,-1.352640,-2.325369,G1,0,3.257356,9.747677,CRC0322_cetux
CRC0322cetux|AAACGCTGTCTGCATA.1|cetux,AAACGCTGTCTGCATA.1,CRC0322,cetux,CRC0322cetux|AAACGCTGTCTGCATA.1|cetux,-1.260545,-2.332163,G1,3,2.957334,8.973439,CRC0322_cetux
...,...,...,...,...,...,...,...,...,...,...,...
CRC0069NT72h|TTTGTTGAGTACAGAT.1|NT72h,TTTGTTGAGTACAGAT.1,CRC0069,NT72h,CRC0069NT72h|TTTGTTGAGTACAGAT.1|NT72h,1.513246,2.504696,G2M,1,13.905034,1.277000,CRC0069_NT72h
CRC0069NT72h|TTTGTTGTCCCAAGCG.1|NT72h,TTTGTTGTCCCAAGCG.1,CRC0069,NT72h,CRC0069NT72h|TTTGTTGTCCCAAGCG.1|NT72h,1.836946,1.123572,S,1,10.575986,3.393476,CRC0069_NT72h
CRC0069NT72h|TTTGTTGTCCGTATAG.1|NT72h,TTTGTTGTCCGTATAG.1,CRC0069,NT72h,CRC0069NT72h|TTTGTTGTCCGTATAG.1|NT72h,0.507208,2.171694,G2M,5,8.119332,5.672054,CRC0069_NT72h


In [47]:
path_to_save='/mnt/cold2/snaketree/prj/PPH/local/share/data/integrated_umap/'
for file in os.listdir(data_path):
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]
    trattamento_sample=sample_name+'_'+trattamento
    if trattamento_sample=='CRC0322_NT':
        nome_file="_".join(str.split(file,sep='_')[4:7])+'_3000_umap_integrated.csv'
    else:
        nome_file="_".join(str.split(file,sep='_')[4:7])+'_umap_integrated.csv'
    if sample_name in train_id and trattamento in tratt_cercato:
        print(trattamento_sample)
        print(nome_file)
        tmp=data_obs[data_obs['trattamento_sample']==trattamento_sample]
        cols=['cell_id','x','y']
        tmp=tmp.loc[:,cols].reset_index(drop=True)
        file=path_to_save+nome_file
        tmp.to_csv(file)
        

CRC0322_cetux
CRC0322_cetux_1_umap_integrated.csv
CRC0322_NT
CRC0322_NT_1_3000_umap_integrated.csv
CRC0327_cetux
CRC0327_cetux_2_umap_integrated.csv
CRC0327_NT
CRC0327_NT_2_umap_integrated.csv
CRC0542_CTX72h
CRC0542_CTX72h_1_umap_integrated.csv
CRC0542_NT72h
CRC0542_NT72h_1_umap_integrated.csv
CRC0069_CTX72h
CRC0069_CTX72h_1_umap_integrated.csv
CRC0069_NT72h
CRC0069_NT72h_1_umap_integrated.csv
